# SMILES 2026 Method 0 Colab Runner

This notebook mounts Google Drive, updates the GitHub repo even if an old copy already exists on disk, installs dependencies, and runs the Method 0 diagnostics.

Recommended Colab runtime: `GPU`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

In [ ]:
import os
import subprocess
from pathlib import Path

TARGET_FOLDER = Path('/content/drive/MyDrive/hallucination_detection')
REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_draft.git'
REPO_NAME = 'hallucination_detection_draft'
REPO_PATH = TARGET_FOLDER / REPO_NAME
AUTO_STASH = True

TARGET_FOLDER.mkdir(parents=True, exist_ok=True)

def run(cmd: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result

print('TARGET_FOLDER =', TARGET_FOLDER)
print('REPO_PATH =', REPO_PATH)

In [ ]:
if not REPO_PATH.exists():
    run(f'git clone {REPO_URL}', cwd=TARGET_FOLDER)

run('git remote -v', cwd=REPO_PATH)
run('git branch --show-current', cwd=REPO_PATH)
run('git fetch origin', cwd=REPO_PATH)
status = run('git status --short', cwd=REPO_PATH, check=False).stdout.strip()

if status:
    print('Local changes detected.')
    if AUTO_STASH:
        run('git stash push -u -m "colab-auto-stash"', cwd=REPO_PATH)
    else:
        raise RuntimeError('Repo is dirty. Set AUTO_STASH = True or clean it manually.')

run('git pull --ff-only origin main', cwd=REPO_PATH)
run('git log --oneline -1', cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('cwd =', Path.cwd())

In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)

In [ ]:
import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Switch Colab runtime to GPU.')

## Method 0 runs

Start with the smoke test. If that completes cleanly, run the full dataset cell.

Current safe defaults for Colab are:

- `--batch-size 2`
- `--cache-dtype float16`
- `--max-length 512`

In [ ]:
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

run(
    'python method0_diagnostics.py '
    '--subset 80 '
    '--batch-size 2 '
    '--cache-dtype float16 '
    '--max-length 512 '
    '--overwrite-cache',
    cwd=REPO_PATH,
)

In [ ]:
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

run(
    'python method0_diagnostics.py '
    '--batch-size 2 '
    '--cache-dtype float16 '
    '--max-length 512 '
    '--overwrite-cache',
    cwd=REPO_PATH,
)

In [ ]:
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

run(
    'python method0_diagnostics.py '
    '--batch-size 4 '
    '--cache-dtype float16 '
    '--max-length 512 '
    '--overwrite-cache',
    cwd=REPO_PATH,
)

In [ ]:
from IPython.display import Image, display
import pandas as pd

artifact_dir = REPO_PATH / 'artifacts' / 'method0'
print('artifact_dir =', artifact_dir)

for name in [
    'silhouette_heatmap.png',
    'silhouette_trends.png',
    'top_pca_scatter.png',
]:
    path = artifact_dir / name
    print(path, 'exists =' , path.exists())
    if path.exists():
        display(Image(filename=str(path)))

top_pairs_path = artifact_dir / 'top_pairs.csv'
if top_pairs_path.exists():
    display(pd.read_csv(top_pairs_path).head(12))